In [1]:
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import ast
import numpy as np


c:\Users\rober\UCD\pol42050\new-sound-of-politics\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:

# Load data
bt_sentences = pd.read_csv("C:/Users/rober/UCD/pol42050/new-sound-of-politics/data/bundestag_sentences20.csv", index_col=False)
# len(bt_speeches["Speech"])
# bt_speeches20 = bt_speeches[bt_speeches["Period"] == 20]
# bt_speeches20.to_csv("C:/Users/rober/UCD/pol42050/new-sound-of-politics/data/bundestag_sentences20.csv")
bt_sentences.head()

,speech_id,sentence_position,sentence,MPID,Date,Chair,Session,Period,Party,Name
0,0,1,Herr Präsident!,17826,2022-01-14,False,12,20,['afd'],Albrecht Glaser
1,0,2,Meine sehr verehrten Damen und Herren!,17826,2022-01-14,False,12,20,['afd'],Albrecht Glaser
2,0,3,Mehr Fortschritt soll es im Lande geben.,17826,2022-01-14,False,12,20,['afd'],Albrecht Glaser
3,0,4,"Was ist das, und wie soll das gehen?",17826,2022-01-14,False,12,20,['afd'],Albrecht Glaser
4,1,1,"Sehr geehrter Herr Dr. Hahn, ich versichere, d...",17826,2022-01-28,False,15,20,['afd'],Albrecht Glaser


In [23]:
# bt_speeches = bt_speeches.sample(frac=0.001)

# bt_speeches['Speech'] = bt_speeches['Speech'].apply(ast.literal_eval)



In [24]:
# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("luerhard/PopBERT")
model = AutoModelForSequenceClassification.from_pretrained("luerhard/PopBERT")

# CPU inference only
device = torch.device("cpu")
model = model.to(device)
model.eval()


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 4431.96it/s]


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31102, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-23): 24 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNorm): LayerNorm((1024,

In [25]:

# Optional: let PyTorch use available CPU threads
# Try 4 first, then 6 or 8 and benchmark on your machine
torch.set_num_threads(6)

# Dynamic INT8 quantization for linear layers
quantized_model = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear},
    dtype=torch.qint8
)
quantized_model.eval()


C:\Users\rober\AppData\Local\Temp\ipykernel_18832\3820193431.py:6: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31102, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-23): 24 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): DynamicQuantizedLinear(in_features=1024, out_features=1024, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
              (key): DynamicQuantizedLinear(in_features=1024, out_features=1024, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
              (value): DynamicQuantizedLinear(in_features=1024, out_features=1024, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
              (dropout): Dropout(p=0.1, inplace=False)
  

In [26]:
len(bt_sentences["sentence"])

44501

In [27]:
subset = bt_sentences  # Process only the first 1000 sentences for testing

sentence_scores = []
for i, sentence in enumerate(subset["sentence"]):
    print(f"analysing sentence {i+1}")
    
    encodings = tokenizer(
        sentence,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    encodings = {k: v.to(device) for k, v in encodings.items()}

    with torch.inference_mode():
        out = quantized_model(**encodings)

    probs = torch.sigmoid(out.logits)
    sentence_scores.append(
        [bt_sentences.at[subset.index[i], "speech_id"], 
        bt_sentences.at[subset.index[i], "sentence_position"]] + 
        probs.cpu().numpy()[0].tolist()
    )


analysing sentence 1
analysing sentence 2
analysing sentence 3
analysing sentence 4
analysing sentence 5
analysing sentence 6
analysing sentence 7
analysing sentence 8
analysing sentence 9
analysing sentence 10
analysing sentence 11
analysing sentence 12
analysing sentence 13
analysing sentence 14
analysing sentence 15
analysing sentence 16
analysing sentence 17
analysing sentence 18
analysing sentence 19
analysing sentence 20
analysing sentence 21
analysing sentence 22
analysing sentence 23
analysing sentence 24
analysing sentence 25
analysing sentence 26
analysing sentence 27
analysing sentence 28
analysing sentence 29
analysing sentence 30
analysing sentence 31
analysing sentence 32
analysing sentence 33
analysing sentence 34
analysing sentence 35
analysing sentence 36
analysing sentence 37
analysing sentence 38
analysing sentence 39
analysing sentence 40
analysing sentence 41
analysing sentence 42
analysing sentence 43
analysing sentence 44
analysing sentence 45
analysing sentence 

KeyboardInterrupt: 

In [ ]:
subset["scores"] = sentence_scores


In [ ]:

# Create DataFrame with custom column names
scores_df = pd.DataFrame(sentence_scores, columns=['speech_id', 'sentence_position', 'Anti-Elitism', 'People-Centrism', 'Left-Wing Host-Ideology', 'Right-Wing Host-Ideology'])


In [ ]:
scores_df.to_csv("populist_scores_per_sentence.csv")

In [ ]:
all_data_combined = pd.merge(subset, scores_df, on=['speech_id', 'sentence_position'])

In [ ]:
all_data_combined.to_csv("all_data_combined.csv", index=False)

,speech_id,sentence_position,sentence,MPID,Date,Chair,Session,Period,Party,Name,scores,Anti-Elitism,People-Centrism,Left-Wing Host-Ideology,Right-Wing Host-Ideology
0,0,1,Herr Präsident!,17826,2022-01-14,False,12,20,['afd'],Albrecht Glaser,"[0, 1, 0.1335628777742386, 0.07978134602308273...",0.133563,0.079781,0.057177,0.071662
1,0,2,Meine sehr verehrten Damen und Herren!,17826,2022-01-14,False,12,20,['afd'],Albrecht Glaser,"[0, 2, 0.14240780472755432, 0.0802032798528671...",0.142408,0.080203,0.056765,0.067925
2,0,3,Mehr Fortschritt soll es im Lande geben.,17826,2022-01-14,False,12,20,['afd'],Albrecht Glaser,"[0, 3, 0.17971497774124146, 0.1364448815584182...",0.179715,0.136445,0.084267,0.174430
3,0,4,"Was ist das, und wie soll das gehen?",17826,2022-01-14,False,12,20,['afd'],Albrecht Glaser,"[0, 4, 0.12137646228075027, 0.0820993185043335...",0.121376,0.082099,0.059250,0.081766
4,1,1,"Sehr geehrter Herr Dr. Hahn, ich versichere, d...",17826,2022-01-28,False,15,20,['afd'],Albrecht Glaser,"[1, 1, 0.15323865413665771, 0.0973944216966629...",0.153239,0.097394,0.064412,0.088444
5,1,2,Ich stelle nur eine einzige Frage an Sie,17826,2022-01-28,False,15,20,['afd'],Albrecht Glaser,"[1, 2, 0.11689471453428268, 0.0686059892177581...",0.116895,0.068606,0.054694,0.067936
6,2,1,Herr Präsident!,17826,2022-04-08,False,29,20,['afd'],Albrecht Glaser,"[2, 1, 0.1335628777742386, 0.07978134602308273...",0.133563,0.079781,0.057177,0.071662
7,2,2,Meine Damen und Herren!,17826,2022-04-08,False,29,20,['afd'],Albrecht Glaser,"[2, 2, 0.14925704896450043, 0.0711181014776229...",0.149257,0.071118,0.064535,0.069349
8,2,3,Verehrter Herr Finanzminister!,17826,2022-04-08,False,29,20,['afd'],Albrecht Glaser,"[2, 3, 0.1510973572731018, 0.069115549325943, ...",0.151097,0.069116,0.062389,0.069242
9,3,1,Frau Präsidentin!,17826,2022-05-13,False,35,20,['afd'],Albrecht Glaser,"[3, 1, 0.1590043604373932, 0.0861503854393959,...",0.159004,0.086150,0.080210,0.072853
